**Organizar Información Agrícola**

En este cuaderno estructuro la información de los siguientes datos para manejarlos en STATA:
- UPRA 2019-2024 y UPRA 2007-2018

# Setup

In [1]:
# define root path of project 
from pathlib import Path
import sys

ROOT = Path("..").resolve()
sys.path.append(str(ROOT))

# load general setup
from utils.setup_general import *

Setup general cargado


# Definir datos de salida
El archivo de salida se estructura para conservar la misma estructura entre todas las bases de datos

In [2]:
ORDEN_DF

['codigo_dane_municipio',
 'anno',
 'nombre_variable',
 'variable_sujeto',
 'variable_medicion',
 'variable_detalle',
 'variable_descripcion',
 'valor',
 'clasificacion_econometria']

In [3]:
# Definir DF con la estructura acordada para el proyecto
# La variable global ORDEN_DF ya tiene la lista de todas las columnas ordenadas
# Inicializar el DF vacío
df_salida = pd.DataFrame(
    columns=[ORDEN_DF]
)

# UPRA armonizado
Upra antiguo y nuevo armonizados

In [4]:
# Cargar datos
df_UPRA = pd.read_parquet(
    DATA/'intermediate/e1001_panel_cultivos_UPRA.parquet'   
)

In [5]:
# Explorar datos
print(MSC_SEPARADOR, 'Head')
display(df_UPRA.head())

print(MSC_SEPARADOR, 'Info')
display(df_UPRA.info())


-------------------------------- Head


,codigo_dane_municipio,anno,nombre_variable,variable_sujeto,variable_medicion,variable_detalle,variable_descripcion,valor,clasificacion_econometria
0,05001,2006,,Suma de todos los cultivos,area_sembrada_ha,Suma de todos los cultivos,Suma de todos los cultivos: area_sembrada_ha d...,276.00,Resultado
1,05001,2007,,Suma de todos los cultivos,area_sembrada_ha,Suma de todos los cultivos,Suma de todos los cultivos: area_sembrada_ha d...,"2,174.00",Resultado
2,05001,2008,,Suma de todos los cultivos,area_sembrada_ha,Suma de todos los cultivos,Suma de todos los cultivos: area_sembrada_ha d...,"2,211.00",Resultado
3,05001,2009,,Suma de todos los cultivos,area_sembrada_ha,Suma de todos los cultivos,Suma de todos los cultivos: area_sembrada_ha d...,"2,197.00",Resultado
4,05001,2010,,Suma de todos los cultivos,area_sembrada_ha,Suma de todos los cultivos,Suma de todos los cultivos: area_sembrada_ha d...,"2,214.50",Resultado



-------------------------------- Info
<class 'pandas.core.frame.DataFrame'>
Index: 13417002 entries, 0 to 118490
Data columns (total 9 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   codigo_dane_municipio      object 
 1   anno                       int32  
 2   nombre_variable            object 
 3   variable_sujeto            object 
 4   variable_medicion          object 
 5   variable_detalle           object 
 6   variable_descripcion       object 
 7   valor                      float64
 8   clasificacion_econometria  object 
dtypes: float64(1), int32(1), object(7)
memory usage: 972.5+ MB


None

In [6]:
# Explorar repetidos desde que cargo los datos
df_UPRA[df_UPRA.duplicated(subset=[COL_ID_MUNICIPIO, COL_ANNO, COL_VARIABLE_DESCRIPCION],
                  keep=False)]

,codigo_dane_municipio,anno,nombre_variable,variable_sujeto,variable_medicion,variable_detalle,variable_descripcion,valor,clasificacion_econometria


## Dar estructura al DF

In [7]:
# Construyo el nombre_variable compatible con STATA a partir de la información
# de las columnas ['variable_sujeto',	'variable_medicion', 'variable_detalle', 'variable_descripcion']

# hacer copia de los datos
panel_UPRA = df_UPRA.copy()


# Definir el concepto de las variables en 8 caracteres
nombre_variable_concepto = 'prodAgr'



### Definir abreviaciones

In [8]:
# Explorar los nombres de los cultivos
display(panel_UPRA[COL_VARIABLE_SUJETO].unique())

# Definir abreviaciones de máximo 6 caracteres:
abreviaciones_cultivos = {
    'Suma de todos los cultivos' : 'total',
    'Hortalizas' : 'GCHort',
    'Leguminosas' : 'GCLegu',
    'Raíces y tubérculos' : 'GCRaiz',
    'Cultivos tropicales tradicionales' : 'GCTrop',
    'Frutales' : 'GCFrut',
    'Cultivos para condimentos, bebidas medicinales y aromáticas' : 'GCMedc',
    'Cereales' : 'GCCerl',
    'Oleaginosas' : 'GCOlea',
    'Transitorio':'CCTrns',
    'Permanente':'CCPerm',
    'Café' : 'Ccafe',
    'Arroz' : 'Carroz',
    'Maíz' : 'Cmaiz',
    'Palma de aceite' : 'CplmAc',
    'Caña' : 'Ccanna',
    'Plátano' : 'Cpltno',
    'Yuca' : 'Cyuca',
    'Cacao' : 'Ccacao',
    'Papa' : 'Cpapa',
    'Frijol' : 'Cfrijl',
    'Banano' : 'Cbnano',
    'Aguacate' : 'Cagcte',
    'Soya' : 'Csoya',
    'Ñame' : 'Cnname',
    'Mango' : 'Cmango',
    'Otros cítricos' : 'CoCtrc',
    'Arveja' : 'Carvja',
    'Naranja' : 'Cnrnja',
    'Algodón' : 'Calgdn',
    'Piña' : 'Cpinna',
    'Coco' : 'Ccoco',
    'Tomate' : 'Ctmate',
    'Habichuela' : 'Chbchl',
    'Limón' : 'Climon',
    'Ahuyama' : 'Cahyma',
    'Lulo' : 'Clulo',
    'Mora' : 'Cmora',
    'Tomate de árbol' : 'CtArbl',
    'Patilla': 'Cptill'
}

# Aplicar las abreviaciones correspondientes
nombre_variable_sujeto = panel_UPRA[COL_VARIABLE_SUJETO].map(abreviaciones_cultivos)

# Explorar que no hayan quedado observaciones sin clasificar
print(MSC_SEPARADOR + "Conteo de Nans: ", nombre_variable_sujeto.isna().sum())
display(panel_UPRA[nombre_variable_sujeto.isna()])

array(['Suma de todos los cultivos', 'Hortalizas', 'Leguminosas',
       'Raíces y tubérculos', 'Cultivos tropicales tradicionales',
       'Frutales',
       'Cultivos para condimentos, bebidas medicinales y aromáticas',
       'Cereales', 'Oleaginosas', 'Transitorio', 'Permanente', 'Café',
       'Arroz', 'Maíz', 'Palma de aceite', 'Caña', 'Plátano', 'Yuca',
       'Cacao', 'Papa', 'Frijol', 'Banano', 'Aguacate', 'Soya', 'Ñame',
       'Mango', 'Otros cítricos', 'Arveja', 'Naranja', 'Algodón', 'Piña',
       'Coco', 'Tomate', 'Habichuela', 'Limón', 'Ahuyama', 'Lulo', 'Mora',
       'Tomate de árbol', 'Patilla'], dtype=object)


--------------------------------Conteo de Nans:  0


,codigo_dane_municipio,anno,nombre_variable,variable_sujeto,variable_medicion,variable_detalle,variable_descripcion,valor,clasificacion_econometria


In [9]:
# Explorar la variable de medidcion
display(panel_UPRA[COL_VARIABLE_MEDICION].unique())

# Definir diccionario de medicion en 6 caracteres
abreviaciones_medicion = {
    'area_sembrada_ha': 'ASembr',
    'area_cosechada_ha': 'ACosch',
    'produccion_t': 'Produc'
}

# Aplicar las abreviaciones correspondientes
nombre_variable_medicion = panel_UPRA[COL_VARIABLE_MEDICION].map(abreviaciones_medicion)

# Explorar que no hayan quedado observaciones sin clasificar
print(MSC_SEPARADOR + "Conteo de Nans: ", nombre_variable_medicion.isna().sum())
display(panel_UPRA[nombre_variable_medicion.isna()])

array(['area_sembrada_ha', 'area_cosechada_ha', 'produccion_t'],
      dtype=object)


--------------------------------Conteo de Nans:  0


,codigo_dane_municipio,anno,nombre_variable,variable_sujeto,variable_medicion,variable_detalle,variable_descripcion,valor,clasificacion_econometria


In [10]:
# Definir diccionario de unidades en 4 caracteres
abreviaciones_unidades = {
    'area_sembrada_ha': 'ha',
    'area_cosechada_ha': 'ha',
    'produccion_t': 'ton'
}

# Aplicar las abreviaciones correspondientes
nombre_variable_unidades = panel_UPRA[COL_VARIABLE_MEDICION].map(abreviaciones_unidades)

# Explorar que no hayan quedado observaciones sin clasificar
print(MSC_SEPARADOR + "Conteo de Nans: ", nombre_variable_unidades.isna().sum())
display(panel_UPRA[nombre_variable_unidades.isna()])


--------------------------------Conteo de Nans:  0


,codigo_dane_municipio,anno,nombre_variable,variable_sujeto,variable_medicion,variable_detalle,variable_descripcion,valor,clasificacion_econometria


In [11]:
# Explorar columna de Detalle
display(panel_UPRA[COL_VARIABLE_DETALLE].unique())

# Las observaciones sin detalle corresponden a la observación de un cultivo puntual
# Todas las demas corresponden a algun tipo de agregación
display(df_UPRA[df_UPRA[COL_VARIABLE_DETALLE]==''])

# Definir diccionario de abreviaciones en 4 caracteres
abreviaciones_detalle = {
    'Suma de todos los cultivos':'totl', # total del municipio-año
    'Suma del grupo':'grpo', # agergacion en grupo
    'Suma del Ciclo':'cicl', # Agregacion en ciclo
    '':'cltv', # mismo cultivo
    'Grupo cultivo:Hortalizas': 'OHor', # Otros cultivos del grupo Hortalizas
    'Grupo cultivo:Leguminosas': 'OLeg', # Otros cultivos del grupo leguminosas
    'Grupo cultivo:Raíces y tubérculos': 'ORai',
    'Grupo cultivo:Cultivos tropicales tradicionales': 'OTrp',
    'Grupo cultivo:Frutales': 'OFrt',
    'Grupo cultivo:Cultivos para condimentos, bebidas medicinales y aromáticas': 'OMed',
    'Grupo cultivo:Cereales': 'OCer',
    'Grupo cultivo:Oleaginosas': 'OOle',
    'Ciclo cultivo:Transitorio': 'OTrn', # Otros cultivos de ciclo transitorio
    'Ciclo cultivo:Permanente': 'OPer' # Otros cultivos de ciclo permanente
}

# Aplicar las abreviaciones correspondientes
nombre_variable_detalle = panel_UPRA[COL_VARIABLE_DETALLE].map(abreviaciones_detalle)

# Explorar que no hayan quedado observaciones sin clasificar
print(MSC_SEPARADOR + "Conteo de Nans: ", nombre_variable_detalle.isna().sum())
display(panel_UPRA[nombre_variable_detalle.isna()])

array(['Suma de todos los cultivos', 'Suma del grupo', 'Suma del Ciclo',
       '', 'Grupo cultivo:Hortalizas', 'Grupo cultivo:Leguminosas',
       'Grupo cultivo:Raíces y tubérculos',
       'Grupo cultivo:Cultivos tropicales tradicionales',
       'Grupo cultivo:Frutales',
       'Grupo cultivo:Cultivos para condimentos, bebidas medicinales y aromáticas',
       'Grupo cultivo:Cereales', 'Grupo cultivo:Oleaginosas',
       'Ciclo cultivo:Transitorio', 'Ciclo cultivo:Permanente'],
      dtype=object)

,codigo_dane_municipio,anno,nombre_variable,variable_sujeto,variable_medicion,variable_detalle,variable_descripcion,valor,clasificacion_econometria
0,05001,2007,,Café,area_sembrada_ha,,Café: area_sembrada_ha de todos los tipos de Café,"1,078.00",Resultado
1,05001,2008,,Café,area_sembrada_ha,,Café: area_sembrada_ha de todos los tipos de Café,"1,078.00",Resultado
2,05001,2009,,Café,area_sembrada_ha,,Café: area_sembrada_ha de todos los tipos de Café,"1,078.00",Resultado
3,05001,2010,,Café,area_sembrada_ha,,Café: area_sembrada_ha de todos los tipos de Café,"1,078.00",Resultado
4,05001,2011,,Café,area_sembrada_ha,,Café: area_sembrada_ha de todos los tipos de Café,"1,081.00",Resultado
...,...,...,...,...,...,...,...,...,...
7720,99773,2008,,Patilla,produccion_t,,Patilla: produccion_t de todos los tipos de Pa...,78.00,Resultado
7721,99773,2009,,Patilla,produccion_t,,Patilla: produccion_t de todos los tipos de Pa...,102.00,Resultado
7722,99773,2010,,Patilla,produccion_t,,Patilla: produccion_t de todos los tipos de Pa...,275.00,Resultado
7723,99773,2013,,Patilla,produccion_t,,Patilla: produccion_t de todos los tipos de Pa...,84.80,Resultado



--------------------------------Conteo de Nans:  0


,codigo_dane_municipio,anno,nombre_variable,variable_sujeto,variable_medicion,variable_detalle,variable_descripcion,valor,clasificacion_econometria


### Construir nombre de la variable con las abreviaciones

In [12]:
panel_UPRA[COL_NOMBRE_DE_VARIABLE] = (nombre_variable_concepto + '_'
                                      + nombre_variable_sujeto + '_'
                                      + nombre_variable_medicion + '_'
                                      + nombre_variable_detalle + '_'
                                      + nombre_variable_unidades)

In [13]:
# Verificar que no haya duplicados
panel_UPRA[panel_UPRA.duplicated(
    subset=[COL_ID_MUNICIPIO, COL_ANNO, COL_NOMBRE_DE_VARIABLE], keep=False)
          ].sort_values(by=[COL_ID_MUNICIPIO, COL_ANNO])

,codigo_dane_municipio,anno,nombre_variable,variable_sujeto,variable_medicion,variable_detalle,variable_descripcion,valor,clasificacion_econometria


# Organizar panel de salida

In [15]:
# Bases a exportar
df_salida = pd.concat(
    [
        panel_UPRA
    ]
)

In [17]:
# Dar forma al Dataframe para leerlo en STATA
df_salida_STATA = df_salida.pivot(
        index=[COL_ANNO, COL_ID_MUNICIPIO],
        columns=COL_NOMBRE_DE_VARIABLE,
        values=COL_VALOR
    ).reset_index().rename_axis(columns=None)

In [18]:
# Mostrar base de datos de la salida
df_salida[ORDEN_DF].head()

,codigo_dane_municipio,anno,nombre_variable,variable_sujeto,variable_medicion,variable_detalle,variable_descripcion,valor,clasificacion_econometria
0,05001,2006,prodAgr_total_ASembr_totl_ha,Suma de todos los cultivos,area_sembrada_ha,Suma de todos los cultivos,Suma de todos los cultivos: area_sembrada_ha d...,276.00,Resultado
1,05001,2007,prodAgr_total_ASembr_totl_ha,Suma de todos los cultivos,area_sembrada_ha,Suma de todos los cultivos,Suma de todos los cultivos: area_sembrada_ha d...,"2,174.00",Resultado
2,05001,2008,prodAgr_total_ASembr_totl_ha,Suma de todos los cultivos,area_sembrada_ha,Suma de todos los cultivos,Suma de todos los cultivos: area_sembrada_ha d...,"2,211.00",Resultado
3,05001,2009,prodAgr_total_ASembr_totl_ha,Suma de todos los cultivos,area_sembrada_ha,Suma de todos los cultivos,Suma de todos los cultivos: area_sembrada_ha d...,"2,197.00",Resultado
4,05001,2010,prodAgr_total_ASembr_totl_ha,Suma de todos los cultivos,area_sembrada_ha,Suma de todos los cultivos,Suma de todos los cultivos: area_sembrada_ha d...,"2,214.50",Resultado


In [27]:
# Explorar variables disponibles por cultivo
df_salida.groupby([COL_VARIABLE_SUJETO,
                   COL_CLASIFICACION_ECONOMETRIA,
                  COL_VARIABLE_MEDICION])['variable_detalle'].value_counts().reset_index().head(50)

,variable_sujeto,clasificacion_econometria,variable_medicion,variable_detalle,count
0,Aguacate,Control,area_cosechada_ha,Ciclo cultivo:Transitorio,20609
1,Aguacate,Control,area_cosechada_ha,Grupo cultivo:Cereales,19145
2,Aguacate,Control,area_cosechada_ha,Ciclo cultivo:Permanente,18883
3,Aguacate,Control,area_cosechada_ha,Grupo cultivo:Frutales,18318
4,Aguacate,Control,area_cosechada_ha,Grupo cultivo:Raíces y tubérculos,17856
5,Aguacate,Control,area_cosechada_ha,Grupo cultivo:Cultivos tropicales tradicionales,15629
6,Aguacate,Control,area_cosechada_ha,Grupo cultivo:Leguminosas,14520
7,Aguacate,Control,area_cosechada_ha,Grupo cultivo:Hortalizas,13534
8,Aguacate,Control,area_cosechada_ha,Grupo cultivo:Oleaginosas,3416
9,Aguacate,Control,area_cosechada_ha,"Grupo cultivo:Cultivos para condimentos, bebid...",1884


In [28]:
# Exportar base de datos para stata

# en pandas se esta manejando objetos "string" que STATA no soporta. Por eso es necesario convertirlo a "str"
df_stata = df_salida_STATA.copy()
columnas_string = df_stata.select_dtypes(include="string").columns

df_stata[columnas_string] = (
    df_stata[columnas_string]
    .astype(object)
    .where(df_stata[columnas_string].notna(), None)
)
df_stata.to_stata(DATA/'intermediate/e1101_panel_informacionAgricola.dta')

In [29]:
# Exportar base de datos en parquet
df_salida.to_parquet(DATA/'intermediate/e1101_panel_informacionAgricola.parquet')
